# NHS Emergency Hospital Admissions — COVID-19 Lockdown Impact
## COMP4037 Research Methods | Coursework 2

---

### Research Question
> **Which NHS emergency hospital admission categories were most suppressed by the COVID-19 lockdown (2020/21), and did they recover to pre-pandemic levels by 2022/23?**

---

### Dataset
| Property | Detail |
|---|---|
| **Source** | NHS England Hospital Episode Statistics (HES) |
| **Sheet** | Primary Diagnosis Summary |
| **Abstraction Level** | ICD-10 Chapter (Primary Diagnosis Summary Level) |
| **Financial Years** | 2018/19 – 2022/23 |
| **Population** | All ages, all genders, England |
| **Metric** | Emergency Hospital Admissions (FAE) |
| **Baseline** | 2018/19 financial year |

---

### Visual Design
| Property | Detail |
|---|---|
| **Type** | Interactive Annotated Diverging Heatmap |
| **Tool** | Python 3 + Plotly (standalone HTML) |
| **X-axis** | Financial year (2019/20 – 2022/23) |
| **Y-axis** | ICD-10 chapter, sorted by 2020/21 lockdown impact |
| **Colour** | Diverging scale — red = decrease, blue = increase vs 2018/19 |
| **Interaction** | 3 dropdown filters + hover tooltips |

---

### Interactive Filters
- **① Sort Order** — Sort chapters by lockdown impact / alphabetically / worst recovery
- **② Drop Threshold** — Show only chapters that dropped >5%, >10%, or >20% during lockdown
- **③ Year Focus** — Isolate a single year column, hiding all others

---

### How to Run
1. Run **Cell 1** to install dependencies
2. Run **Cell 2** to load data and generate the visualization
3. Run **Cell 3** to display the interactive chart inline
4. Click the **camera icon** (top-right of chart) to export a high-resolution PNG

### Visual Diagram on github:

> **https://mohammedadnan04.github.io/Research_Methods_CW2/**
---


In [ ]:
# ── Install dependencies & download data automatically ───
!pip install plotly openpyxl -q

import os
import gdown

# Your public Google Drive folder ID
FOLDER_ID = "1N7z20wEeiuI7-PEqGPs6OJ08iAD5QzGh"

# Download entire folder into /content/NHS_Data
os.makedirs('/content/NHS_Data', exist_ok=True)
gdown.download_folder(
    id=FOLDER_ID,
    output='/content/NHS_Data',
    quiet=False,
    use_cookies=False
)

print("\n✅ Data downloaded successfully")
print("Files:", os.listdir('/content/NHS_Data'))

Retrieving folder contents


Processing file 1qbeHzBe_Ae-gvvq2ufLX4iKuTEukICgC hosp-epis-stat-admi-diag-2018-19-tab.xlsx
Processing file 1ZJ_Yc84j--fsgP_Nc4NAvzm2hEyDIg1Y hosp-epis-stat-admi-diag-2019-20-tab supp.xlsx
Processing file 1GAdp7voJIcQgBJG2j7IUPCmoQ-bc-Jy- hosp-epis-stat-admi-diag-2020-21-tab.xlsx
Processing file 13pbNWpY72fIBUxPSV19p5mXy2dGUTp2_ hosp-epis-stat-admi-diag-2021-22-tab.xlsx
Processing file 1o9ECprA_0_sYbTAhOIpgxT1KIfMPO0mm hosp-epis-stat-admi-diag-2022-23-tab_V2.xlsx


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1qbeHzBe_Ae-gvvq2ufLX4iKuTEukICgC
To: /content/NHS_Data/hosp-epis-stat-admi-diag-2018-19-tab.xlsx
100%|██████████| 6.11M/6.11M [00:00<00:00, 135MB/s]
Downloading...
From: https://drive.google.com/uc?id=1ZJ_Yc84j--fsgP_Nc4NAvzm2hEyDIg1Y
To: /content/NHS_Data/hosp-epis-stat-admi-diag-2019-20-tab supp.xlsx
100%|██████████| 6.00M/6.00M [00:00<00:00, 69.9MB/s]
Downloading...
From: https://drive.google.com/uc?id=1GAdp7voJIcQgBJG2j7IUPCmoQ-bc-Jy-
To: /content/NHS_Data/hosp-epis-stat-admi-diag-2020-21-tab.xlsx
100%|██████████| 5.88M/5.88M [00:00<00:00, 126MB/s]
Downloading...
From: https://drive.google.com/uc?id=13pbNWpY72fIBUxPSV19p5mXy2dGUTp2_
To: /content/NHS_Data/hosp-epis-stat-admi-diag-2021-22-tab.xlsx
100%|██████████| 5.95M/5.95M [00:00<00:00, 90.1MB/s]
Downloading...
From: https://drive.google.com/uc?id=1o9ECprA_0_sYbTAhOIpgxT1KIfM


✅ Data downloaded successfully
Files: ['hosp-epis-stat-admi-diag-2020-21-tab.xlsx', 'hosp-epis-stat-admi-diag-2022-23-tab_V2.xlsx', 'hosp-epis-stat-admi-diag-2019-20-tab supp.xlsx', 'hosp-epis-stat-admi-diag-2018-19-tab.xlsx', 'hosp-epis-stat-admi-diag-2021-22-tab.xlsx']



Download completed


In [ ]:
# =============================================================
# CELL 2 — Load data and generate visualization
# =============================================================

import openpyxl
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

# ── File paths ───────────────────────────────────────────
# ── File paths ───────────────────────────────────────────
FILES = {
    "2018/19": "/content/NHS_Data/hosp-epis-stat-admi-diag-2018-19-tab.xlsx",
    "2019/20": "/content/NHS_Data/hosp-epis-stat-admi-diag-2019-20-tab supp.xlsx",
    "2020/21": "/content/NHS_Data/hosp-epis-stat-admi-diag-2020-21-tab.xlsx",
    "2021/22": "/content/NHS_Data/hosp-epis-stat-admi-diag-2021-22-tab.xlsx",
    "2022/23": "/content/NHS_Data/hosp-epis-stat-admi-diag-2022-23-tab_V2.xlsx",
}

# Header row index differs between older and newer file formats
HEADER_IDX = {
    "2018/19": 10, "2019/20": 10,
    "2020/21": 9,  "2021/22": 9, "2022/23": 9,
}

AGE_COLS = [
    'Age 0','Age 1-4','Age 5-9','Age 10-14',
    'Age 15','Age 16','Age 17','Age 18','Age 19',
    'Age 20-24','Age 25-29','Age 30-34','Age 35-39',
    'Age 40-44','Age 45-49','Age 50-54','Age 55-59',
    'Age 60-64','Age 65-69','Age 70-74','Age 75-79',
    'Age 80-84','Age 85-89','Age 90+'
]

# Broader age bands used in hover tooltips
AGE_BANDS = {
    '0–4':   ['Age 0','Age 1-4'],
    '5–14':  ['Age 5-9','Age 10-14'],
    '15–19': ['Age 15','Age 16','Age 17','Age 18','Age 19'],
    '20–29': ['Age 20-24','Age 25-29'],
    '30–39': ['Age 30-34','Age 35-39'],
    '40–49': ['Age 40-44','Age 45-49'],
    '50–59': ['Age 50-54','Age 55-59'],
    '60–69': ['Age 60-64','Age 65-69'],
    '70–79': ['Age 70-74','Age 75-79'],
    '80+':   ['Age 80-84','Age 85-89','Age 90+'],
}


def get_chapter(code):
    """
    Map ICD-10 summary code to chapter name using first-letter prefix.
    Works for both broad codes (e.g. E15-E90 in 2018/19) and granular codes
    (e.g. E15-E16, E20-E35 in later files) which differ across NHS reporting years.
    """
    if not code or not str(code)[0].isalpha():
        return None
    L = code[0].upper()
    mapping = {
        'A': 'Infectious & Parasitic', 'B': 'Infectious & Parasitic',
        'C': 'Neoplasms (Cancer)',
        'E': 'Endocrine & Metabolic',  'F': 'Mental & Behavioural',
        'G': 'Nervous System',         'H': 'Eye & Ear',
        'I': 'Circulatory',            'J': 'Respiratory',
        'K': 'Digestive',              'L': 'Skin & Tissue',
        'M': 'Musculoskeletal',        'N': 'Genitourinary',
        'O': 'Pregnancy & Childbirth', 'P': 'Perinatal',
        'Q': 'Congenital Anomalies',   'R': 'Symptoms & Signs',
        'S': 'Injuries & Poisoning',   'T': 'Injuries & Poisoning',
        'U': 'Infectious & Parasitic', 'Z': 'Health Service Contact',
    }
    if L == 'D':
        # D00–D48 = Neoplasms, D50+ = Blood & Immune disorders
        try:
            return 'Neoplasms (Cancer)' if int(code[1:3]) <= 48 else 'Blood & Immune'
        except:
            return 'Neoplasms (Cancer)'
    return mapping.get(L, None)


def load_year(year, path):
    """Load the Primary Diagnosis Summary sheet for one financial year."""
    wb   = openpyxl.load_workbook(path, read_only=True)
    ws   = wb['Primary Diagnosis Summary']
    rows = list(ws.iter_rows(values_only=True))

    header  = rows[HEADER_IDX[year]]
    col_map = {str(h).strip(): j for j, h in enumerate(header)
               if h is not None and str(h).strip() != ''}

    def safe(row, col, default=0.0):
        idx = col_map.get(col)
        if idx is None: return default
        v = row[idx]
        if v is None or v in ('', '*'): return default
        try: return float(v)
        except: return default

    records = []
    for row in rows[HEADER_IDX[year] + 2:]:
        code = row[0]
        if not code or str(code).strip() == '': continue
        code_str = str(code).strip()
        if any(kw in code_str for kw in
               ['Copyright','Disclosure','Responsible','Contact','protect']):
            break
        if code_str == 'Total' or not code_str[0].isalpha():
            continue
        rec = {
            'year': year, 'code': code_str,
            'description': str(row[1]).strip() if row[1] else '',
            'admissions': safe(row, 'Admissions'),
            'emergency':  safe(row, 'Emergency'),
        }
        for age in AGE_COLS:
            rec[age] = safe(row, age)
        records.append(rec)

    return pd.DataFrame(records)


# ── Load all 5 files ─────────────────────────────────────
print("Loading files...")
dfs = []
for year, path in FILES.items():
    df = load_year(year, path)
    dfs.append(df)
    print(f"  {year}: {len(df)} codes | Emergency total: {df['emergency'].sum():,.0f}")

df_all = pd.concat(dfs, ignore_index=True)
df_all['chapter'] = df_all['code'].apply(get_chapter)
df_all = df_all[df_all['chapter'].notna()].copy()

# Aggregate individual age columns into broader bands
for band, cols in AGE_BANDS.items():
    existing = [c for c in cols if c in df_all.columns]
    df_all[f'ageband_{band}'] = df_all[existing].sum(axis=1)

BAND_COLS        = [f'ageband_{b}' for b in AGE_BANDS]
DISPLAY_YEARS    = ['2019/20','2020/21','2021/22','2022/23']
EXCLUDE_CHAPTERS = ['Pregnancy & Childbirth']  # No 2018/19 baseline available
BANDS            = list(AGE_BANDS.keys())


# ── Aggregate to chapter × year ──────────────────────────
chapter_year = df_all.groupby(['chapter','year']).agg(
    emergency  = ('emergency','sum'),
    admissions = ('admissions','sum'),
    **{bc: (bc,'sum') for bc in BAND_COLS}
).reset_index()

# Compute % change vs 2018/19 baseline
baseline = (chapter_year[chapter_year['year'] == '2018/19']
            [['chapter','emergency']]
            .rename(columns={'emergency':'baseline_emerg'}))
chapter_year = chapter_year.merge(baseline, on='chapter', how='left')
chapter_year['pct_change'] = np.where(
    chapter_year['baseline_emerg'] > 0,
    (chapter_year['emergency'] - chapter_year['baseline_emerg'])
    / chapter_year['baseline_emerg'] * 100, np.nan)

# Pivot to chapter × year matrix (exclude baseline year column — always 0%)
heat_base = (chapter_year[~chapter_year['chapter'].isin(EXCLUDE_CHAPTERS)]
             .pivot_table(index='chapter', columns='year', values='pct_change')
             .reindex(columns=DISPLAY_YEARS))

# Sort orders for Filter ①
order_lockdown = list(heat_base.sort_values('2020/21', ascending=True).index)
order_alpha    = sorted(order_lockdown)
order_recovery = list(heat_base.sort_values('2022/23', ascending=True).index)

# Thresholds for Filter ② — matched to actual data distribution
THRESHOLDS = [0, 5, 10, 20]
THRESH_LABELS = {
    0:  'All chapters  (18)',
    5:  'Dropped >5% in lockdown  (11 chapters)',
    10: 'Dropped >10% in lockdown  (8 chapters)',
    20: 'Dropped >20% in lockdown  (2 chapters)',
}

def filter_chapters(order, thresh):
    if thresh == 0: return order
    filtered = [c for c in order
                if c in heat_base.index
                and not pd.isna(heat_base.loc[c, '2020/21'])
                and heat_base.loc[c, '2020/21'] <= -thresh]
    return filtered if filtered else order


# ── Hover tooltip builder ─────────────────────────────────
def build_hover(chapters, years):
    hover = []
    for chapter in chapters:
        row_h = []
        for year in years:
            rd = chapter_year[
                (chapter_year['chapter'] == chapter) &
                (chapter_year['year']    == year)]
            if rd.empty:
                row_h.append(f'<b>{chapter}</b><br>{year}: No data')
                continue
            r         = rd.iloc[0]
            pct       = r['pct_change']
            pct_str   = f'{pct:+.1f}%' if not np.isnan(pct) else 'N/A'
            direction = '▼ decrease' if pct < 0 else '▲ increase'
            age_vals  = {b: r.get(f'ageband_{b}', 0) for b in BANDS}
            top3      = sorted(age_vals.items(), key=lambda x: x[1], reverse=True)[:3]
            age_str   = '  |  '.join([f'<b>Age {b}:</b> {int(v):,}' for b, v in top3])
            row_h.append(
                f'<b>{chapter}</b><br>'
                f'────────────────────────<br>'
                f'Year: <b>{year}</b><br>'
                f'Emergency Admissions: <b>{int(r["emergency"]):,}</b><br>'
                f'Change vs 2018/19: <b>{pct_str}</b> {direction}<br>'
                f'Baseline (2018/19): {int(r["baseline_emerg"]):,}<br>'
                f'Total Admissions: {int(r["admissions"]):,}<br>'
                f'────────────────────────<br>'
                f'Top Age Groups:<br>{age_str}'
            )
        hover.append(row_h)
    return hover


def build_z_text(chapters, years):
    z, text = [], []
    for chapter in chapters:
        z_row, t_row = [], []
        for year in years:
            v = heat_base.loc[chapter, year] if chapter in heat_base.index else np.nan
            z_row.append(v)
            t_row.append('N/A' if pd.isna(v) else f'{v:+.0f}%')
        z.append(z_row)
        text.append(t_row)
    return z, text


# ── Colour palette ────────────────────────────────────────
BG      = '#111827'
CARD    = '#1F2937'
GRID    = '#374151'
TEXT    = '#F9FAFB'
SUBTEXT = '#9CA3AF'
ACCENT  = '#60A5FA'

YEAR_COLS = {
    '2019/20': '#38BDF8',
    '2020/21': '#F87171',
    '2021/22': '#FB923C',
    '2022/23': '#34D399',
}

YEAR_LABELS = {
    '2019/20': 'Pre-COVID',
    '2020/21': '🔒 COVID Lockdown',
    '2021/22': 'Recovery Year 1',
    '2022/23': 'Recovery Year 2',
}

# Diverging scale: vivid red (large drop) → dark neutral (0%) → vivid blue (large rise)
DIV_SCALE = [
    [0.00, '#7F1D1D'], [0.15, '#DC2626'], [0.28, '#F87171'],
    [0.42, '#FCA5A5'], [0.50, '#374151'], [0.58, '#93C5FD'],
    [0.72, '#3B82F6'], [0.85, '#1D4ED8'], [1.00, '#1E3A5F'],
]


# ── Build all 60 traces ───────────────────────────────────
# 3 sort orders × 4 thresholds × 5 year focus options = 60 traces
# Pre-computing all combinations enables instant filter switching
YEAR_FOCUS_OPTIONS = [None] + DISPLAY_YEARS
traces     = []
trace_meta = []

def make_trace(chapters, years, visible=False):
    z, text = build_z_text(chapters, years)
    hover   = build_hover(chapters, years)
    return go.Heatmap(
        z=z, x=years, y=chapters,
        text=text,
        hovertext=hover,
        hovertemplate='%{hovertext}<extra></extra>',
        texttemplate='%{text}',
        textfont=dict(size=13, color=TEXT, family='Inter, Arial'),
        colorscale=DIV_SCALE,
        zmid=0, zmin=-70, zmax=70,
        colorbar=dict(
            title=dict(
                text='% Change vs 2018/19 Baseline',
                font=dict(color=SUBTEXT, size=11),
                side='right',
            ),
            tickfont=dict(color=SUBTEXT, size=10),
            tickvals=[-60,-40,-20,0,20,40,60],
            ticktext=['-60%','-40%','-20%','0','+20%','+40%','+60%'],
            thickness=16, len=0.72, x=1.02,
            outlinecolor=GRID, outlinewidth=1,
            bgcolor=CARD,
        ),
        xgap=5, ygap=5,
        visible=visible,
    )

for sort_key, order in [
    ('lockdown', order_lockdown),
    ('alpha',    order_alpha),
    ('recovery', order_recovery),
]:
    for thresh in THRESHOLDS:
        chapters = filter_chapters(order, thresh)
        for yr_focus in YEAR_FOCUS_OPTIONS:
            years   = [yr_focus] if yr_focus else DISPLAY_YEARS
            default = (sort_key == 'lockdown' and thresh == 0 and yr_focus is None)
            traces.append(make_trace(chapters, years, visible=default))
            trace_meta.append((sort_key, thresh, yr_focus))

fig = go.Figure(data=traces)


def vis(sort_key, thresh, yr_focus=None):
    """Return visibility boolean list for the matching trace."""
    return [(s == sort_key and t == thresh and y == yr_focus)
            for s, t, y in trace_meta]


# ── Annotation builders ───────────────────────────────────
def make_base_annotations():
    """Permanent annotations: filter labels, provenance strip, observation box."""
    anns = []

    for x, y, label, color in [
        (0.00, 1.234, '① Sort Order', '#60A5FA'),
        (0.57, 1.234, '② Drop Threshold  (based on 2020/21 lockdown)', '#FB923C'),
        (0.00, 1.136, '③ Year Focus', '#34D399'),
    ]:
        anns.append(dict(
            x=x, y=y, xref='paper', yref='paper',
            text=f'<b>{label}</b>', showarrow=False,
            font=dict(color=color, size=10, family='Inter, Arial'),
            align='left',
        ))

    anns.append(dict(
        x=0.5, y=1.31, xref='paper', yref='paper',
        text=(
            '<b>Data Level:</b> Primary Diagnosis Summary (ICD-10 Chapter)  ·  '
            '<b>Population:</b> All ages, all genders, England  ·  '
            '<b>Metric:</b> Emergency Admissions (FAE)  ·  '
            '<b>Baseline:</b> 2018/19  ·  '
            '<b>Source:</b> NHS England HES'
        ),
        showarrow=False,
        font=dict(color=SUBTEXT, size=10, family='Inter, Arial'),
        align='center',
    ))

    anns.append(dict(
        x=0.5, y=-0.23, xref='paper', yref='paper',
        text=(
            '<b>🔍 Unique Observation:</b> '
            'Respiratory admissions fell <b>-63%</b> in 2020/21 - the largest collapse across all ICD-10 chapters.<br>'
            'Infectious & Parasitic (-44%) and Injuries & Poisoning (-14%) also dropped sharply during lockdown.<br>'
            'Critically, <b>Mental & Behavioural</b> disorders show a <b>persistent decline reaching -31% by 2022/23</b>, '
            'indicating a structural long-term shift rather than a temporary lockdown effect.'
        ),
        showarrow=False,
        font=dict(color=TEXT, size=11, family='Inter, Arial'),
        align='center',
        bgcolor='rgba(17,24,39,0.97)',
        bordercolor=ACCENT, borderwidth=1.5,
        borderpad=14, width=920,
    ))

    return anns


def make_year_label_anns(years):
    """Column header labels - built only for the years currently shown."""
    return [
        dict(
            x=yr, y=1.045,
            xref='x', yref='paper',
            text=f'<b>{YEAR_LABELS[yr]}</b>',
            showarrow=False,
            font=dict(color=YEAR_COLS[yr], size=11, family='Inter, Arial'),
            align='center',
        )
        for yr in years
    ]


BASE_ANNOTATIONS    = make_base_annotations()
initial_annotations = BASE_ANNOTATIONS + make_year_label_anns(DISPLAY_YEARS)


# ── Filter buttons ────────────────────────────────────────
# Each button updates both trace visibility and annotations (so year labels update too)

sort_buttons = [
    dict(label='📉  Lockdown Impact  (worst first)',
         method='update',
         args=[{'visible': vis('lockdown', 0, None)},
               {'annotations': BASE_ANNOTATIONS + make_year_label_anns(DISPLAY_YEARS)}]),
    dict(label='🔤  Alphabetical  (A → Z)',
         method='update',
         args=[{'visible': vis('alpha', 0, None)},
               {'annotations': BASE_ANNOTATIONS + make_year_label_anns(DISPLAY_YEARS)}]),
    dict(label='📈  Worst Recovery  (2022/23 worst first)',
         method='update',
         args=[{'visible': vis('recovery', 0, None)},
               {'annotations': BASE_ANNOTATIONS + make_year_label_anns(DISPLAY_YEARS)}]),
]

thresh_buttons = [
    dict(label=THRESH_LABELS[t],
         method='update',
         args=[{'visible': vis('lockdown', t, None)},
               {'annotations': BASE_ANNOTATIONS + make_year_label_anns(DISPLAY_YEARS)}])
    for t in THRESHOLDS
]

year_focus_buttons = [
    dict(label='🗓  All years',
         method='update',
         args=[{'visible': vis('lockdown', 0, None)},
               {'annotations': BASE_ANNOTATIONS + make_year_label_anns(DISPLAY_YEARS)}]),
    dict(label='📘  2019/20  -  Pre-COVID',
         method='update',
         args=[{'visible': vis('lockdown', 0, '2019/20')},
               {'annotations': BASE_ANNOTATIONS + make_year_label_anns(['2019/20'])}]),
    dict(label='🔒  2020/21  -  COVID Lockdown',
         method='update',
         args=[{'visible': vis('lockdown', 0, '2020/21')},
               {'annotations': BASE_ANNOTATIONS + make_year_label_anns(['2020/21'])}]),
    dict(label='📙  2021/22  -  Recovery Year 1',
         method='update',
         args=[{'visible': vis('lockdown', 0, '2021/22')},
               {'annotations': BASE_ANNOTATIONS + make_year_label_anns(['2021/22'])}]),
    dict(label='📗  2022/23  -  Recovery Year 2',
         method='update',
         args=[{'visible': vis('lockdown', 0, '2022/23')},
               {'annotations': BASE_ANNOTATIONS + make_year_label_anns(['2022/23'])}]),
]


def menu(buttons, x, y, border_color, active=0):
    return dict(
        buttons=buttons,
        direction='down', showactive=True,
        x=x, y=y, xanchor='left', yanchor='top',
        bgcolor=CARD,
        bordercolor=border_color, borderwidth=1.5,
        font=dict(color='#dbdb1a', size=11, family='Inter, Arial'),
        active=active,
        pad=dict(r=12, t=5, b=5),
    )

fig.update_layout(
    updatemenus=[
        menu(sort_buttons,       x=0.00, y=1.22, border_color='#60A5FA'),
        menu(thresh_buttons,     x=0.44, y=1.22, border_color='#FB923C'),
        menu(year_focus_buttons, x=0.00, y=1.12, border_color='#34D399'),
    ],
    shapes=[],
)

# ── Layout ───────────────────────────────────────────────
fig.update_layout(
    title=dict(
        text='<b>NHS Emergency Hospital Admissions - COVID-19 Lockdown Impact by Diagnosis</b>',
        x=0.5, xanchor='center',
        font=dict(size=18, color=TEXT, family='Inter, Arial'),
        pad=dict(t=10), y=0.985,
    ),
    height=1120,
    width=1320,
    paper_bgcolor=BG,
    plot_bgcolor=CARD,
    font=dict(family='Inter, Arial, sans-serif', color=TEXT, size=11),
    xaxis=dict(
        title=dict(text='Financial Year', font=dict(color=SUBTEXT, size=12)),
        tickfont=dict(color=TEXT, size=13, family='Inter, Arial'),
        gridcolor=GRID, linecolor=GRID,
        side='bottom', ticklen=0,
    ),
    yaxis=dict(
        title=dict(
            text='ICD-10 Diagnostic Chapter  (Primary Diagnosis Summary Level)',
            font=dict(color=SUBTEXT, size=11)),
        tickfont=dict(color=TEXT, size=12, family='Inter, Arial'),
        gridcolor=GRID, linecolor=GRID,
        autorange='reversed', ticklen=0,
    ),
    margin=dict(l=260, r=185, t=250, b=230),
    annotations=initial_annotations,
    hoverlabel=dict(
        bgcolor=CARD, bordercolor=ACCENT,
        font=dict(color=TEXT, size=12, family='Inter, Arial'),
        namelength=-1,
    ),
)

# ── Save HTML ─────────────────────────────────────────────
output_path = '/content/NHS_COVID_Interactive_Final.html'
fig.write_html(
    output_path,
    include_plotlyjs='cdn',
    config={
        'displayModeBar': True,
        'modeBarButtonsToRemove': ['select2d','lasso2d','autoScale2d'],
        'toImageButtonOptions': {
            'format': 'png',
            'filename': 'NHS_COVID_Lockdown_Heatmap',
            'height': 1120, 'width': 1320, 'scale': 3,
        },
        'responsive': True,
    }
)

# Plotly hardcodes white as the dropdown hover colour in its bundled JS
# Replace those values directly in the saved HTML to match our dark theme
with open(output_path, 'r') as f:
    html = f.read()

for old, new in [
    ('"fff"', '"1F2937"'), ("'fff'", "'1F2937'"),
    ('"#fff"', '"#1F2937"'), ("'#fff'", "'#1F2937'"),
    ('"white"', '"#1F2937"'), ("'white'", "'#1F2937'"),
]:
    html = html.replace(old, new)

with open(output_path, 'w') as f:
    f.write(html)

print("✅ Saved:", output_path)
print("\nFilters:")
print("  ① Sort Order     — lockdown impact / alphabetical / worst recovery")
print("  ② Drop Threshold — all (18) / >5% (11) / >10% (8) / >20% (2)")
print("  ③ Year Focus     — isolates one year, column label updates automatically")

fig.show()


Loading files...
  2018/19: 140 codes | Emergency total: 2,040,721
  2019/20: 206 codes | Emergency total: 2,130,747
  2020/21: 206 codes | Emergency total: 1,716,628
  2021/22: 215 codes | Emergency total: 2,215,655
  2022/23: 215 codes | Emergency total: 2,194,962
✅ Saved: /content/NHS_COVID_Interactive_Final.html

Filters:
  ① Sort Order     — lockdown impact / alphabetical / worst recovery
  ② Drop Threshold — all (18) / >5% (11) / >10% (8) / >20% (2)
  ③ Year Focus     — isolates one year, column label updates automatically
